In [1]:
from gensim.models.fasttext import load_facebook_vectors
import spacy

path = "cc.pl.300.bin"

model = load_facebook_vectors(path)

nlp = spacy.load("pl_core_news_lg")

In [2]:
def get_similar_words(word, topn=10):
    try:
        similar = model.most_similar(word, topn=topn)
        return [w for w, score in similar]
    except KeyError:
        return []
    
def filter_by_pos(word, candidates):
    word_doc = nlp(word)
    if not word_doc:
        return []
    
    word_pos = word_doc[0].pos_
    filtered = []
    
    for cand_doc in nlp.pipe(candidates):
        if cand_doc and cand_doc[0].pos_ == word_pos:
            filtered.append(cand_doc.text)
    return filtered

In [3]:
import re
import simplemma

def lemmatize(word):
    return simplemma.lemmatize(word, lang="pl")

def group_similar_words_word2vec(text, threshold=0.75):
    doc = nlp(text.lower())
    
    token_map = {token.text: token for token in doc if len(token.text) >= 3}
    words_in_text = set(token_map.keys())

    grouped = {}
    seen = set()

    for w in words_in_text:
        if w in seen or w not in model:
            continue
        
        token_w = token_map[w]
        
        raw_similar = model.most_similar(w, topn=30)
        
        filtered_group = []
        for cand_word, score in raw_similar:
            if cand_word not in words_in_text or score < threshold:
                continue
            
            token_cand = token_map[cand_word]
            
            if token_w.pos_ != token_cand.pos_:
                continue
            
            if token_w.lemma_ == token_cand.lemma_:
                continue
            
            if token_cand.is_stop:
                continue

            filtered_group.append(cand_word)

        if filtered_group:
            grouped[w] = sorted(list(set(filtered_group)))
            seen.update(filtered_group)
            seen.add(w)

    return grouped

In [4]:
with open("tekst.txt", "r", encoding="utf-8") as f:
    tekst = f.read()

wynik = group_similar_words_word2vec(tekst)

for rep, group in wynik.items():
    print(f"{rep}: {group}")

with open("results_spaCy.txt", "w", encoding="utf-8") as out:
    for rep, group in wynik.items():
        out.write(f"{rep}: {group}\n")


miało: ['mogło']
drugie: ['pierwsze']
2009: ['2002', '2003', '2008', '2010', '2012', '2013', '2014']
zobligowany: ['zobowiązany']
jednej: ['drugiej']
kwotę: ['sumę']
przedłożenie: ['przedłożenia']
określeniu: ['ustaleniu']
rozstrzygnięcie: ['rozstrzygnięcia']
2008r: ['2005r', '2010r', '2012r']
września: ['grudnia', 'lipca', 'lutego', 'maja', 'października', 'sierpnia']
przedstawienia: ['przedstawienie']
wystąpiły: ['występowały']
zakwestionował: ['kwestionował']
2015: ['2010', '2012', '2013', '2014']
wskazał: ['stwierdził']
pierwszym: ['kolejnym']
powiedzieć: ['mówić']
wtedy: ['wówczas']
wymienionych: ['wskazanych']
odpowiedni: ['właściwy']
podkreślił: ['stwierdził']
109: ['122']
wyniosła: ['wynosiła']
wyniku: ['skutek']
sprawa: ['kwestia']
potwierdził: ['stwierdził']
między: ['pomiędzy']
gdyż: ['albowiem']
wypadku: ['przypadku']
miała: ['mogła']
nieuzasadnione: ['uzasadnione']
tej: ['takiej']
zasądza: ['zasądził']
tego: ['takiego']
także: ['ponadto']
wpływa: ['oddziałuje']
354: ['353'